# exp007: A vs. B nas 15 categorias do MVTec AD (Colab)

Experimento intermediário antes do Artigo 1. 740 imagens (50 por categoria, 40 no `toothbrush`), 3 perguntas, Configurações A e B.

Funciona no Colab pelo navegador e com kernel do Colab no VS Code. No navegador, os resultados vão para o Drive. No VS Code o `drive.mount` não funciona, então eles ficam na VM, em `/content/exp007_storage`. Nos dois casos, a última célula embute os resultados num zip dentro da própria saída do notebook. Salve o `.ipynb` e, no repo local, rode `python3 -m src.unpack_colab_results notebooks/03_exp007_multicategoria_colab.ipynb`.

Use GPU. L4 e A100 rodam em bf16; a T4 cai para fp16.

In [2]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE = '/content/drive/MyDrive/ic-visionexplain'
except Exception as error:
    # Kernel do Colab no VS Code não monta o Drive; os arquivos ficam na VM.
    print('Drive indisponível:', error)
    STORAGE = '/content/exp007_storage'
os.makedirs(f'{STORAGE}/exp007_multicategoria', exist_ok=True)
os.makedirs(f'{STORAGE}/anomalib_runs', exist_ok=True)
print('Resultados em', STORAGE)

Mounted at /content/drive
Resultados em /content/drive/MyDrive/ic-visionexplain


In [3]:
%cd /content
!test -d vision-explain || git clone https://github.com/Edryck/vision-explain.git
%cd /content/vision-explain
!git pull --ff-only
!pip install -q -r requirements.txt

/content
Cloning into 'vision-explain'...
remote: Enumerating objects: 235, done.
remote: Counting objects: 100% (235/235), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 235 (delta 99), reused 232 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (235/235), 5.64 MiB | 36.35 MiB/s, done.
Resolving deltas: 100% (99/99), done.
/content/vision-explain
Already up to date.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Dados

Se o Drive montou, suba `mvtec-ad.zip` para `MyDrive/ic-visionexplain/`. Sem zip, o anomalib baixa o MVTec sozinho no primeiro treino (mais lento).

In [ ]:
!mkdir -p data/mvtec_ad
if not os.path.isdir('data/mvtec_ad/mvtec-ad/bottle') and os.path.exists(f'{STORAGE}/mvtec-ad.zip'):
    !cp {STORAGE}/mvtec-ad.zip /content/mvtec-ad.zip
    !unzip -q /content/mvtec-ad.zip -d data/mvtec_ad/mvtec-ad/
!ls data/mvtec_ad/mvtec-ad | head -20

ls: cannot access 'data/mvtec_ad/mvtec-ad': No such file or directory


Resultados e checkpoints do PatchCore ficam em `STORAGE` via link simbólico. O `subset.json` do repo só é copiado se ainda não houver um lá.

In [ ]:
!test -f {STORAGE}/exp007_multicategoria/subset.json || cp experiments/exp007_multicategoria/subset.json {STORAGE}/exp007_multicategoria/
!rm -rf experiments/exp007_multicategoria data/anomalib_runs
!ln -s {STORAGE}/exp007_multicategoria experiments/exp007_multicategoria
!ln -s {STORAGE}/anomalib_runs data/anomalib_runs
!ls experiments/exp007_multicategoria

import subprocess, sys

def run(*args):
    # Com '!', um erro não para o notebook e as células seguintes rodam sobre arquivo faltando.
    subprocess.run([sys.executable, '-m', *args], check=True)

cp: cannot stat 'experiments/exp007_multicategoria/subset.json': No such file or directory


In [ ]:
import torch
print(torch.tpu.get_device_name(0))

TPU v5e


## 1. Evidências do PatchCore

Treina um PatchCore por categoria (coreset 0,01, igual ao bottle), roda o teste completo (AUROC de imagem e de pixel) e extrai a evidência das imagens do subconjunto.

In [ ]:
run('src.configB.exp007_multicategoria.evidences')

CalledProcessError: Command '['/usr/bin/python3', '-m', 'src.configB.exp007_multicategoria.evidences']' returned non-zero exit status 1.

## 2. Configuração A (imagem + pergunta)

In [ ]:
run('src.configA.exp007_multicategoria.run')

## 3. Configuração B (imagem + evidência + pergunta)

In [ ]:
run('src.configB.exp007_multicategoria.run')

## 4. Métricas

Uma célula por passo, para saber qual falhou. O FaithScore carrega o SmolVLM de novo; se faltar memória, reinicie o kernel e rode só as células de métricas.

In [ ]:
run('src.multicategory_metrics', 'score', '--config', 'A', '--faithscore')

In [ ]:
run('src.multicategory_metrics', 'score', '--config', 'B', '--faithscore')

In [ ]:
run('src.multicategory_metrics', 'report')

In [ ]:
import pandas as pd
pd.read_csv('experiments/exp007_multicategoria/results/tabela_por_categoria.csv')

In [ ]:
import json
summary = json.load(open('experiments/exp007_multicategoria/results/resumo_exp007.json'))
print(json.dumps({'overall': summary['overall'], 'paired': summary['paired']}, indent=1)[:6000])

## 5. Exportar resultados

Rode esta célula mesmo que as métricas tenham falhado: ela leva o que existir. Depois salve o notebook (Ctrl+S no VS Code; no navegador, Arquivo > Baixar .ipynb) e, no repo local, rode:

```
python3 -m src.unpack_colab_results notebooks/03_exp007_multicategoria_colab.ipynb
```

Os checkpoints do PatchCore (`anomalib_runs`) ficam de fora porque são pesados.

In [ ]:
import base64, io, zipfile
from pathlib import Path

source = Path('experiments/exp007_multicategoria').resolve()
buffer = io.BytesIO()
with zipfile.ZipFile(buffer, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(source.rglob('*')):
        if path.is_file() and path.suffix != '.tmp':
            archive.write(path, path.relative_to(source).as_posix())
            print(path.relative_to(source))
encoded = base64.b64encode(buffer.getvalue()).decode()
print(f'{len(buffer.getvalue()) / 1e6:.1f} MB')
# Marcadores lidos pelo src/unpack_colab_results.py.
print('EXP007_ZIP_BEGIN')
print(encoded)
print('EXP007_ZIP_END')